### Week 6 – Apache Spark Assignment

#### Technologies
- Apache Spark
- PySpark
- Databricks

#### Objective
To understand Spark Architecture, DataFrames, Lazy Evaluation,
Transformations, Actions, Fault Tolerance, Parquet processing,
and performance optimization using Apache Spark.


##### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark Application

Apache Spark follows a **Master-Worker Architecture** where different components work together to execute a distributed application efficiently.

---

###### Spark Architecture

```
                   User Application
                          │
                          ▼
                  +----------------+
                  |     Driver     |
                  +----------------+
                          │
          Requests Resources from Cluster Manager
                          │
                          ▼
               +-----------------------+
               |   Cluster Manager     |
               +-----------------------+
                 │         │         │
                 ▼         ▼         ▼
          +----------+ +----------+ +----------+
          |Executor 1| |Executor 2| |Executor 3|
          +----------+ +----------+ +----------+
             │              │             │
          Task 1         Task 2       Task 3
             │              │             │
             └──────────────┴─────────────┘
                        Results
                          │
                          ▼
                        Driver
```

---

###### 1. Driver

The **Driver** is the main process of every Spark application.

It is responsible for:

- Creating the SparkSession
- Reading user-written PySpark code
- Building the Logical Execution Plan
- Optimizing the query using the Catalyst Optimizer
- Creating the DAG (Directed Acyclic Graph)
- Dividing work into stages and tasks
- Sending tasks to Executors
- Collecting results from Executors

**Think of the Driver as the brain of the Spark application.**

---

###### 2. Cluster Manager

The **Cluster Manager** is responsible for managing the resources of the cluster.

Its responsibilities include:

- Allocating CPU cores and memory
- Launching Executor processes
- Monitoring available resources
- Scheduling resources for Spark applications

Spark can work with multiple Cluster Managers such as:

- Standalone Cluster Manager
- YARN
- Kubernetes
- Apache Mesos

**Think of the Cluster Manager as the resource manager of the cluster.**

---

####### 3. Executor

Executors are worker processes that perform the actual computation.

Each Executor:

- Receives tasks from the Driver
- Executes transformations and actions
- Stores cached data in memory
- Performs shuffle operations
- Returns results back to the Driver

Executors continue running throughout the application's lifetime unless the application finishes or they fail.

**Think of Executors as the workers that perform all computations.**

---

####### Execution Flow

1. User submits a Spark application.
2. The Driver starts and creates a SparkSession.
3. The Driver requests resources from the Cluster Manager.
4. The Cluster Manager launches multiple Executors.
5. The Driver divides the application into stages and tasks.
6. Tasks are distributed among Executors.
7. Executors process data in parallel.
8. Results are returned to the Driver.
9. The Driver sends the final output to the user.

---

####### Responsibilities Summary

| Component | Primary Responsibility |
|------------|------------------------|
| Driver | Controls the entire Spark application, creates execution plans, schedules tasks, and collects results |
| Cluster Manager | Allocates cluster resources and launches Executors |
| Executor | Executes tasks, stores intermediate data, performs computations, and returns results |

---

####### Production Insight

In Databricks, the **Driver node** coordinates the application, while multiple **Worker nodes** run Executors. This distributed architecture enables Spark to process massive datasets efficiently by executing tasks in parallel across multiple machines.

---

###### Key Takeaway

> **Driver = Brain of the application**  
> **Cluster Manager = Resource allocator**  
> **Executors = Workers that execute tasks**

Together, these three components enable Apache Spark to perform fast, scalable, and fault-tolerant distributed data processing.


##### Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

###### What is Lazy Evaluation?

**Lazy Evaluation** is one of the core optimization techniques used by Apache Spark.

Instead of executing every transformation immediately, Spark records all transformations and creates a Logical Execution Plan (DAG). The actual execution begins only when an Action (such as `show()`, `count()`, `collect()`, or `write()`) is called.

This allows Spark to optimize the complete workflow before processing any data.

---

###### How Lazy Evaluation Works

Suppose we write the following transformations:

df.filter(...)
  .select(...)
  .withColumn(...)

Spark does not execute these operations one by one.

Instead, it stores them as a logical plan.

Read CSV
      │
      ▼
Filter Rows
      │
      ▼
Select Columns
      │
      ▼
Add New Column
      │
      ▼
Logical Plan (DAG)
      │
      ▼
Catalyst Optimizer
      │
      ▼
Optimized Physical Plan
      │
      ▼
Execution (Only after an Action)

---

##### Why Lazy Evaluation Improves Performance

###### 1. Query Optimization

Spark combines multiple transformations into a single optimized execution plan instead of executing each transformation separately.

---

###### 2. Reduced Disk I/O

Since intermediate results are not written after every transformation, unnecessary read/write operations are avoided.

---

###### 3. Reduced Data Movement

Spark analyzes the entire DAG and minimizes unnecessary shuffle operations, reducing network communication between executors.

---

###### 4. Better Resource Utilization

Only the transformations required to produce the final result are executed, saving CPU and memory resources.

---

##### Example

Without Lazy Evaluation:

Read Data
↓

Filter

↓

Execute

↓

Select

↓

Execute

↓

Add Column

↓

Execute

Multiple executions increase processing time.

---

With Lazy Evaluation:

Read Data

↓

Filter

↓

Select

↓

Add Column

↓

Create DAG

↓

Optimize

↓

Single Execution

Spark performs all operations together after optimization.

---

###### Production Insight

In large-scale ETL pipelines processing terabytes of data, a Spark job may contain dozens of transformations. Lazy Evaluation allows Spark to optimize the complete pipeline before execution, significantly reducing execution time, disk I/O, and network communication.

---

##### Key Takeaway

> Spark does not execute transformations immediately.
> It first builds a DAG, optimizes the entire execution plan using the Catalyst Optimizer, and executes only when an Action is called. This optimization makes Spark much faster than systems that execute every operation immediately.


##### Q3: Read a CSV File with Header and Schema Inference

###### Objective

The objective is to load a CSV dataset into a Spark DataFrame while treating the first row as the column header and automatically inferring the data types of each column.

---

###### What is a CSV File?

CSV (Comma-Separated Values) is one of the most commonly used file formats for storing structured data. Each row represents a record, and each column is separated by commas.

Spark can efficiently read CSV files and convert them into distributed DataFrames for further processing.

---

###### Parameters Used

###### header=True
Treats the first row of the CSV file as column names instead of data values.

###### inferSchema=True
Automatically detects the data type of each column (Integer, Double, String, Date, etc.), eliminating the need to manually define the schema.

---

###### Business Scenario

Consider an e-commerce company that receives daily sales data in CSV format. Before performing data cleaning and analysis, the dataset must be loaded into Spark with the correct column names and data types.

Using header=True and inferSchema=True ensures that the data is immediately ready for transformation and analytics.

---

###### Production Insight

While inferSchema=True is useful during development and data exploration, production ETL pipelines usually define the schema explicitly using StructType for better performance and consistency.

In [0]:
df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)
print("CSV FILE LOADED SUCCESSFULLY")
display(df)

CSV FILE LOADED SUCCESSFULLY


order_id,user_id,product_id,category,region,priority,status,base_price,price,amount,old_name
ORD00001,U1163,P0170,Clothing,South,High,Cancelled,557.35,584.18,2786.75,Aditya Joshi
ORD00002,U1108,P0123,Clothing,South,Low,Cancelled,181.35,170.43,906.75,Myra Reddy
ORD00003,U1114,P0101,Clothing,West,Medium,Pending,819.62,894.57,2458.86,Sai Verma
ORD00004,U1097,P0188,Toys,East,High,Cancelled,2324.15,2149.76,9296.6,Aditya Mehta
ORD00005,U1075,P0192,Toys,South,Low,Completed,276.83,261.76,830.49,Aditya Rao
ORD00006,U1059,P0197,Grocery,West,Low,Pending,855.14,830.39,2565.42,Aditya Joshi
ORD00007,U1162,P0162,Clothing,West,Medium,Pending,4948.14,5086.69,24740.7,Arjun Kapoor
ORD00008,U1083,P0114,Clothing,North,Medium,Pending,1375.32,1295.82,6876.6,Saanvi Reddy
ORD00009,U1167,P0217,Clothing,East,High,Completed,3737.7,3766.84,18688.5,Myra Bose
ORD00010,U1149,P0156,Clothing,West,High,Completed,4312.46,4013.04,8624.92,Myra Joshi


In [0]:
print("Schema of Dataset")
df.printSchema()

print("\nTotal Rows :", df.count())
print("Total Columns :", len(df.columns))

print("\nColumn Names")
print(df.columns)

# Display first 5 records
df.show(5, truncate=False)

Schema of Dataset
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- old_name: string (nullable = true)


Total Rows : 1000
Total Columns : 11

Column Names
['order_id', 'user_id', 'product_id', 'category', 'region', 'priority', 'status', 'base_price', 'price', 'amount', 'old_name']
+--------+-------+----------+--------+------+--------+---------+----------+-------+-------+------------+
|order_id|user_id|product_id|category|region|priority|status   |base_price|price  |amount |old_name    |
+--------+-------+----------+--------+------+--------+---------+----------+-------+-------+------------+
|ORD00001|U1163  |P0170     |Clothing|South |Hig

###### Observation

- The CSV file was successfully loaded into a Spark DataFrame.
- The first row was correctly interpreted as the column names because header=True was used.
- Spark automatically inferred the appropriate data types for each column using inferSchema=True.
- The dataset is now ready for data cleaning, transformation, and analytical operations.

---

###### Key Takeaway

> Spark's read.csv() function provides a simple and efficient way to load structured CSV data into a distributed DataFrame. Using header=True improves readability, while inferSchema=True simplifies exploratory data analysis by automatically detecting column data types.


##### Q4: Difference Between CSV and Parquet in Terms of Storage and Performance

###### What are CSV and Parquet?

Both CSV and Parquet are file formats used to store structured data. However, they organize data differently, which has a significant impact on storage efficiency and query performance.

---

###### CSV (Row-Based Storage)

CSV stores data row by row. Every row contains values for all columns.

###### Example

| Product_ID | Category | Price | Quantity |
|------------|----------|-------|----------|
| P101 | Electronics | 25000 | 2 |
| P102 | Furniture | 18000 | 1 |
| P103 | Electronics | 35000 | 3 |

Stored internally as:

P101, Electronics, 25000, 2
P102, Furniture, 18000, 1
P103, Electronics, 35000, 3

When Spark reads only the Price column, it still has to scan every column of every row.

---

###### Parquet (Columnar Storage)

Parquet stores data column by column instead of row by row.

###### Product_ID
-----------
P101
P102
P103

###### Category
-----------
Electronics
Furniture
Electronics

###### Price
-----------
25000
18000
35000

###### Quantity
-----------
2
1
3

If Spark needs only the Price column, it reads only that column instead of the entire dataset.

---

###### Why Does This Matter?

Suppose a dataset contains 100 columns, but an analytics query requires only Price and Quantity.

###### CSV

Spark reads all 100 columns from disk before selecting the required columns.

Read 100 Columns
↓

Select 2 Columns

---

###### Parquet

Spark directly reads only the required columns.

Read Price

+

Read Quantity

↓

Process Data

This significantly reduces disk I/O and memory usage.

---

###### Performance Comparison

| Feature | CSV | Parquet |
|----------|-----|----------|
| Storage Format | Row-Based | Columnar |
| File Size | Larger | Smaller (Compressed) |
| Compression | No | Yes |
| Read Speed | Slower | Faster |
| Query Performance | Lower | Higher |
| Schema Support | Basic | Rich Schema |
| Best For | Data Exchange | Analytics & Big Data |

---

###### Advantages of Parquet

- Reads only required columns (Column Pruning)
- Supports built-in compression
- Reduces disk I/O
- Improves query execution speed
- Optimized for Apache Spark
- Supports Predicate Pushdown for faster filtering

---

###### Real-World Example

Imagine an e-commerce company storing 500 GB of sales data.

A business analyst wants only the Price column.

###### Using CSV

Spark reads the entire 500 GB dataset before selecting the Price column.

###### Using Parquet

Spark reads only the Price column, which may require reading only a small fraction of the data.

As a result, query execution becomes significantly faster while using less memory and CPU resources.

---

###### Production Insight

Modern data engineering platforms such as Databricks, Azure Data Lake, AWS EMR, Snowflake, and Apache Spark** primarily use Parquet because it provides better compression, faster query execution, and efficient storage for large-scale analytical workloads.

---

###### Key Takeaway

> CSV is designed for simple data exchange, whereas Parquet is designed for high-performance analytics.
> Because Parquet stores data column-wise, Spark reads only the required columns, resulting in faster queries, lower memory usage, and improved overall performance.

In [0]:
df_csv = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

print("CSV Dataset Loaded Successfully")
print("Rows :", df_csv.count())
print("Columns :", len(df_csv.columns))


# Load the parquet file
df_csv.write.mode("overwrite").parquet(
    "path/to/input"
)

print("Parquet file created successfully.")

# Read the parquet file
df_parquet = spark.read.parquet(
    "path/to/input"
)

display(df_parquet)


CSV Dataset Loaded Successfully
Rows : 1000
Columns : 11
Parquet file created successfully.


order_id,user_id,product_id,category,region,priority,status,base_price,price,amount,old_name
ORD00001,U1163,P0170,Clothing,South,High,Cancelled,557.35,584.18,2786.75,Aditya Joshi
ORD00002,U1108,P0123,Clothing,South,Low,Cancelled,181.35,170.43,906.75,Myra Reddy
ORD00003,U1114,P0101,Clothing,West,Medium,Pending,819.62,894.57,2458.86,Sai Verma
ORD00004,U1097,P0188,Toys,East,High,Cancelled,2324.15,2149.76,9296.6,Aditya Mehta
ORD00005,U1075,P0192,Toys,South,Low,Completed,276.83,261.76,830.49,Aditya Rao
ORD00006,U1059,P0197,Grocery,West,Low,Pending,855.14,830.39,2565.42,Aditya Joshi
ORD00007,U1162,P0162,Clothing,West,Medium,Pending,4948.14,5086.69,24740.7,Arjun Kapoor
ORD00008,U1083,P0114,Clothing,North,Medium,Pending,1375.32,1295.82,6876.6,Saanvi Reddy
ORD00009,U1167,P0217,Clothing,East,High,Completed,3737.7,3766.84,18688.5,Myra Bose
ORD00010,U1149,P0156,Clothing,West,High,Completed,4312.46,4013.04,8624.92,Myra Joshi


##### Observation

- The dataset was successfully converted from CSV to Parquet format.
- Both files contain the same data, but Parquet stores the data in a columnar format.
- Columnar storage enables Spark to read only the required columns, reducing disk I/O and improving query performance.
- Due to built-in compression and optimized storage, Parquet is the preferred format for large-scale data engineering and analytics workloads.


##### Q5: Select `product_id` and `price` where the category is **'Electronics'**

###### Objective

The objective is to filter the dataset and retrieve only the products that belong to the **Electronics** category while displaying only the **product_id** and **price** columns.

---

###### Spark Concepts Used

###### filter()
Filters rows based on a given condition.

###### select()
Retrieves only the required columns from the DataFrame.

---

###### Business Scenario

Suppose an e-commerce company wants to analyze only electronic products for pricing analysis.

Instead of processing the complete dataset, Spark filters only the required category and selects only the necessary columns. This reduces unnecessary computation and improves query efficiency.

---

###### Production Insight

Selecting only the required columns before performing further transformations reduces memory usage and improves performance, especially when working with large datasets containing hundreds of columns.

In [0]:
from pyspark.sql.functions import col

electronics_df = (
    df.filter(col("category") == "Electronics")
      .select("product_id", "price")
)

display(electronics_df)

print("Validation")
print("Total Electronics Products :", electronics_df.count())
print("\nSchema")
electronics_df.printSchema()

product_id,price
P0120,3555.42
P0158,2480.1
P0238,2054.06
P0111,4568.35
P0128,4250.25
P0107,4588.71
P0240,2212.79
P0238,1810.89
P0224,3346.46
P0236,3054.61


Validation
Total Electronics Products : 178

Schema
root
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)



###### Observation

- The dataset was successfully filtered to include only products belonging to the Electronics category.
- Only the product identifier and price columns were selected, reducing unnecessary data processing.
- Filtering before further analysis improves query efficiency and minimizes memory consumption.

---

###### Key Takeaway

> Spark's filter() and select() operations allow efficient row filtering and column projection. Processing only the required data improves performance and is considered a best practice in large-scale data engineering pipelines.


##### Q6: Rename a Column and Change the Data Type of Another Column

###### Objective

The objective is to revise the DataFrame by:

1. Renaming the column old_name to new_name
2. Converting the price column from StringType to DoubleType

These operations are common during data cleaning and preprocessing to ensure consistency and compatibility for further analysis.

---

###### Spark Concepts Used

###### withColumnRenamed()
Renames an existing column without modifying the original DataFrame.

###### cast()
Converts a column from one data type to another.

---

###### Business Scenario

Suppose an organization receives sales data from multiple sources.

- One source provides a column named old_name, but the standard data model expects new_name.
- The price column is stored as text (StringType), making mathematical calculations impossible.

Before performing analytics, the dataset must be standardized by renaming columns and converting data types.

---

###### Production Insight

Renaming columns improves consistency across different datasets, while proper data types ensure accurate calculations and better performance in Spark SQL and DataFrame operations.

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

print("Original Schema")
df.printSchema()

df_revised = (
    df.withColumnRenamed("old_name", "new_name")
      .withColumn("price", col("price").cast(DoubleType()))
)

print("Revised Schema")

df_revised.printSchema()

Original Schema
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- old_name: string (nullable = true)

Revised Schema
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- new_name: string (nullable = true)



In [0]:
print("Validation")

print("Columns after Renaming:")
print(df_revised.columns)

print("\nPreview of Updated DataFrame:")
display(df_revised.select("new_name", "price"))

Validation
Columns after Renaming:
['order_id', 'user_id', 'product_id', 'category', 'region', 'priority', 'status', 'base_price', 'price', 'amount', 'new_name']

Preview of Updated DataFrame:


new_name,price
Aditya Joshi,584.18
Myra Reddy,170.43
Sai Verma,894.57
Aditya Mehta,2149.76
Aditya Rao,261.76
Aditya Joshi,830.39
Arjun Kapoor,5086.69
Saanvi Reddy,1295.82
Myra Bose,3766.84
Myra Joshi,4013.04


###### Observation

- The column old_name was successfully renamed to new_name.
- The price column was converted from StringType to DoubleType, making it suitable for numerical operations.
- The original DataFrame remained unchanged because Spark DataFrames are immutable.
- A new DataFrame (df_revised) was created containing the updated schema.

---

###### Key Takeaway

> Spark provides simple yet powerful methods such as withColumnRenamed() and cast() to standardize datasets before performing analytics. Data type conversion is an essential preprocessing step because numerical operations cannot be performed reliably on string values.


##### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

###### What is a Lineage Graph (DAG)?

The Lineage Graph, also known as the Directed Acyclic Graph (DAG), is a sequence of transformations that Spark records while processing data.

Instead of storing multiple copies of intermediate data on disk, Spark remembers how each DataFrame or RDD was created. If a partition is lost due to a worker node failure, Spark can rebuild only the lost partition by replaying the required transformations.

This mechanism provides fault tolerance while reducing unnecessary storage overhead.

---

##### How Spark Uses DAG for Fault Tolerance
df = spark.read.csv(...)

filtered_df = df.filter(col("price") > 500)

selected_df = filtered_df.select("product_id", "price")

final_df = selected_df.withColumn(
    "discount_price",
    col("price") * 0.90
)

Spark does not execute these transformations immediately.

Instead, it creates a Lineage Graph (DAG):

Read CSV
    │
    ▼
Filter Records
    │
    ▼
Select Columns
    │
    ▼
Create discount_price
    │
    ▼
Final DataFrame

Each transformation is recorded in the DAG.

---

###### What Happens if a Worker Node Fails?

Suppose one executor stores Partition 3, and that executor crashes.

Executor 1 → Partition 1 

Executor 2 → Partition 2

Executor 3 → Partition 3 (Failed)

Executor 4 → Partition 4 

Instead of restarting the entire job, Spark checks the Lineage Graph and identifies how Partition 3 was created.

Spark then:

1. Launches a new Executor.
2. Re-executes only the transformations required to recreate Partition 3.
3. Continues processing without affecting the remaining partitions.

This minimizes recovery time and avoids unnecessary recomputation.

---

###### Why is this Better than MapReduce?

In Hadoop MapReduce, intermediate data is written to HDFS after every stage for recovery.

Spark follows a different approach:

- Stores transformation history (Lineage Graph)
- Recomputes only lost partitions when required
- Avoids repeated disk writes
- Reduces storage overhead
- Improves overall performance

---

###### Real-World Example

Consider an online retail company processing millions of customer transactions.

During processing, one worker node crashes unexpectedly.

Instead of restarting the complete ETL pipeline, Spark recreates only the missing partition using the DAG and resumes execution. This significantly reduces downtime and improves reliability.

---

###### Production Insight

Spark's Lineage Graph is one of the primary reasons why Spark is both **fast** and **fault tolerant**. It avoids maintaining multiple copies of intermediate data while still ensuring reliable recovery from node failures.

---

###### Key Takeaway

> Spark achieves fault tolerance by maintaining a **Lineage Graph (DAG)** of all transformations. If a worker node fails, Spark reconstructs only the lost partition instead of restarting the entire application, making distributed processing both efficient and reliable.

In [0]:
from pyspark.sql.functions import col

df_lineage = (
    df.filter(col("price") > 500)
      .select("product_id", "category", "price")
      .withColumn("discount_price", col("price") * 0.90)
)
print("Logical and Physical Execution Plan:\n")
df_lineage.explain(True)

Logical and Physical Execution Plan:

== Parsed Logical Plan ==
Project [product_id#11538, category#11539, price#11544, (price#11544 * 0.9) AS discount_price#11550]
+- Project [product_id#11538, category#11539, price#11544]
   +- Filter (price#11544 > cast(500 as double))
      +- Relation [order_id#11536,user_id#11537,product_id#11538,category#11539,region#11540,priority#11541,status#11542,base_price#11543,price#11544,amount#11545,old_name#11546] csv

== Analyzed Logical Plan ==
product_id: string, category: string, price: double, discount_price: double
Project [product_id#11538, category#11539, price#11544, (price#11544 * 0.9) AS discount_price#11550]
+- Project [product_id#11538, category#11539, price#11544]
   +- Filter (price#11544 > cast(500 as double))
      +- Relation [order_id#11536,user_id#11537,product_id#11538,category#11539,region#11540,priority#11541,status#11542,base_price#11543,price#11544,amount#11545,old_name#11546] csv

== Optimized Logical Plan ==
Project [product_

###### Observation

- Spark did not execute each transformation separately.
- It first created a Lineage Graph (DAG) representing all transformations.
- The execution plan generated by explain(True) shows how Spark organizes transformations before execution.
- If any partition is lost due to a worker node failure, Spark can use this DAG to reconstruct only the missing partition instead of restarting the entire application.

---


###### Q8: Filter Orders Where Status is 'Completed' and Amount is Greater Than 1000

###### Objective

The objective is to filter the dataset and retrieve only those orders that satisfy the following conditions:

- The status should be 'Completed'
- The amount should be greater than 1000

This helps identify high-value completed transactions for further analysis.

---

###### Spark Concepts Used

###### filter()

The filter() function is used to retrieve rows that satisfy one or more conditions.

###### Logical AND (`&`)

The AND operator ensures that both conditions must be true for a record to be included in the result.

---

###### Business Scenario

Suppose an e-commerce company wants to generate a report of all successfully completed high-value orders.

Instead of scanning the entire dataset manually, Spark filters only those transactions that meet the required business criteria.

This filtered dataset can be used for:

- Revenue Analysis
- Customer Insights
- Sales Reporting
- High-Value Order Monitoring

---

###### Production Insight

Filtering data as early as possible in an ETL pipeline reduces the amount of data processed in subsequent transformations. This improves query performance, lowers memory usage, and minimizes unnecessary computation.

In [0]:
from pyspark.sql.functions import col

completed_orders = (
    df.filter(
        (col("status") == "Completed") &
        (col("amount") > 1000)
    )
)

print("Completed Orders with Amount > 1000")
display(completed_orders)

Completed Orders with Amount > 1000


order_id,user_id,product_id,category,region,priority,status,base_price,price,amount,old_name
ORD00009,U1167,P0217,Clothing,East,High,Completed,3737.7,3766.84,18688.5,Myra Bose
ORD00010,U1149,P0156,Clothing,West,High,Completed,4312.46,4013.04,8624.92,Myra Joshi
ORD00011,U1016,P0219,Toys,East,Low,Completed,3417.4,3153.95,17087.0,Ananya Malhotra
ORD00012,U1164,P0175,Furniture,South,Medium,Completed,4771.39,5130.06,14314.17,Rohan Malhotra
ORD00017,U1132,P0163,Clothing,North,Medium,Completed,2962.13,2802.24,5924.26,Aarav Verma
ORD00018,U1181,P0158,Electronics,North,Medium,Completed,2595.07,2480.1,10380.28,Kabir Mehta
ORD00022,U1012,P0238,Electronics,North,High,Completed,2061.75,2054.06,8247.0,Vivaan Gupta
ORD00025,U1014,P0120,Clothing,North,Low,Completed,3392.3,3212.65,3392.3,Priya Reddy
ORD00026,U1148,P0120,Furniture,East,Medium,Completed,3365.14,3240.08,10095.42,Kiara Gupta
ORD00032,U1109,P0128,Electronics,South,Low,Completed,4181.25,4250.25,8362.5,Myra Gupta


In [0]:
print("Validation")
print("Total Matching Records :", completed_orders.count())
print("\nSchema")

completed_orders.printSchema()

Validation
Total Matching Records : 305

Schema
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- old_name: string (nullable = true)



In [0]:
display(
    completed_orders.select(
        "order_id",
        "user_id",
        "product_id",
        "amount",
        "status"
    )
)

order_id,user_id,product_id,amount,status
ORD00009,U1167,P0217,18688.5,Completed
ORD00010,U1149,P0156,8624.92,Completed
ORD00011,U1016,P0219,17087.0,Completed
ORD00012,U1164,P0175,14314.17,Completed
ORD00017,U1132,P0163,5924.26,Completed
ORD00018,U1181,P0158,10380.28,Completed
ORD00022,U1012,P0238,8247.0,Completed
ORD00025,U1014,P0120,3392.3,Completed
ORD00026,U1148,P0120,10095.42,Completed
ORD00032,U1109,P0128,8362.5,Completed


In [0]:
print("Execution Plan")
completed_orders.explain()

Execution Plan
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonFilter (((isnotnull(amount#11700) AND isnotnull(status#11697)) AND (amount#11700 > 1000.0)) AND (status#11697 = Completed))
      +- PhotonRowToColumnar
         +- FileScan csv [order_id#11691,user_id#11692,product_id#11693,category#11694,region#11695,priority#11696,status#11697,base_price#11698,price#11699,amount#11700,old_name#11701] Batched: false, DataFilters: [isnotnull(amount#11700), isnotnull(status#11697), (amount#11700 > 1000.0), (status#11697 = Compl..., Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/dbacademy/default/tutorials/source.csv], PartitionFilters: [], PushedFilters: [IsNotNull(amount), IsNotNull(status), GreaterThan(amount,1000.0), EqualTo(status,Completed)], ReadSchema: struct<order_id:string,user_id:string,product_id:string,category:string,region:string,priority:st...


== Photon Explanation ==
The query is fully supported by Photon.


###### Observation
- The dataset was successfully filtered using two conditions.
- Only records where status = 'Completed' and amount > 1000 were retained.
- Spark applied both conditions efficiently using the filter() transformation.
- Filtering the data before further processing reduces the dataset size and improves performance for downstream operations.

---

###### Key Takeaway
> Applying filters early in a Spark pipeline is considered a best practice because it reduces unnecessary data processing, improves execution speed, and optimizes resource utilization.

Spark applies Predicate Pushdown (when supported by the data source, such as Parquet), allowing filter conditions to be pushed closer to the storage layer. This reduces the amount of data read into memory and improves query performance.

Although this dataset is currently stored as CSV, the same filter would be significantly faster if the data were stored in Parquet format because Spark could read only the required data.


##### Q9: Explain the Concept of Predicate Pushdown in Parquet and How It Affects the Amount of Data Loaded into Memory

##### What is Predicate Pushdown?

Predicate Pushdown is a query optimization technique used by Apache Spark when reading columnar file formats such as Parquet.

Instead of loading the entire dataset into memory and then applying filter conditions, Spark pushes the filter condition down to the storage layer. As a result, only the required rows are read into memory.

This significantly reduces disk I/O, memory usage, and query execution time.

---

###### How Predicate Pushdown Works

Suppose we execute the following query:
df.filter(col("price") > 1000)

###### Without Predicate Pushdown (CSV)

Read Entire CSV File
        │
        ▼
Load All Rows into Memory
        │
        ▼
Apply Filter (price > 1000)
        │
        ▼
Return Required Rows

Spark must scan the complete dataset before applying the filter.

###### With Predicate Pushdown (Parquet)

Filter Condition
(price > 1000)
        │
        ▼
Parquet Storage
        │
        ▼
Read Only Matching Data
        │
        ▼
Load Required Rows into Memory

Only the relevant data is read from disk, making execution much faster.

---

###### Why Does It Improve Performance?

Predicate Pushdown provides several performance benefits:

- Reduces disk I/O by reading only the required rows.
- Decreases memory consumption.
- Improves query execution speed.
- Minimizes unnecessary network communication.
- Makes analytical queries more efficient on large datasets.

---

###### Real-World Example

Imagine an online retail company stores 500 GB of sales data in Parquet format.

A data analyst wants to retrieve only transactions where:
price > 1000

Instead of reading the complete 500 GB dataset, Spark reads only the row groups containing matching records. This dramatically reduces the amount of data loaded into memory and speeds up query execution.

---

###### CSV vs Parquet

| Feature | CSV | Parquet |
|----------|-----|----------|
| Predicate Pushdown |  Not Supported |  Supported |
| Data Read | Entire File | Only Matching Data |
| Memory Usage | High | Low |
| Query Performance | Slower | Faster |

---

###### Production Insight

Most modern Data Engineering platforms such as Databricks, Azure Data Lake, AWS EMR, and Snowflake store analytical datasets in Parquet because Predicate Pushdown and Column Pruning significantly improve performance for large-scale analytics.

---

###### Key Takeaway

> Predicate Pushdown allows Spark to apply filter conditions while reading Parquet files, ensuring that only the required data is loaded into memory. This reduces disk I/O, lowers memory usage, and improves overall query performance.

In [0]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("price") > 1000)
print("Products with Price Greater Than 1000")

display(filtered_df)

Products with Price Greater Than 1000


order_id,user_id,product_id,category,region,priority,status,base_price,price,amount,old_name
ORD00004,U1097,P0188,Toys,East,High,Cancelled,2324.15,2149.76,9296.6,Aditya Mehta
ORD00007,U1162,P0162,Clothing,West,Medium,Pending,4948.14,5086.69,24740.7,Arjun Kapoor
ORD00008,U1083,P0114,Clothing,North,Medium,Pending,1375.32,1295.82,6876.6,Saanvi Reddy
ORD00009,U1167,P0217,Clothing,East,High,Completed,3737.7,3766.84,18688.5,Myra Bose
ORD00010,U1149,P0156,Clothing,West,High,Completed,4312.46,4013.04,8624.92,Myra Joshi
ORD00011,U1016,P0219,Toys,East,Low,Completed,3417.4,3153.95,17087.0,Ananya Malhotra
ORD00012,U1164,P0175,Furniture,South,Medium,Completed,4771.39,5130.06,14314.17,Rohan Malhotra
ORD00013,U1045,P0127,Books,East,Low,Cancelled,3064.4,2851.63,6128.8,Neha Malhotra
ORD00014,U1135,P0182,Furniture,North,High,Pending,4399.67,4691.51,13199.01,Arjun Sharma
ORD00015,U1061,P0120,Electronics,West,High,Cancelled,3840.88,3555.42,15363.52,Neha Gupta


In [0]:
print("Validation")
print("Total Matching Records :", filtered_df.count())
filtered_df.printSchema()

Validation
Total Matching Records : 811
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- old_name: string (nullable = true)



###### Observation

- The dataset was successfully filtered using the specified condition.
- Since the current dataset is stored as CSV, Spark reads the entire file before applying the filter.
- If the same dataset were stored in Parquet, Spark could use Predicate Pushdown to read only the matching records, resulting in better performance and lower memory usage.

---

###### Interview Insight

Predicate Pushdown is one of the key reasons why Parquet is preferred over CSV for analytical workloads.

Combined with Column Pruning, it enables Spark to read only the required rows and columns, making large-scale data processing significantly faster.


##### Q10: Add a New Column `final_price` by Applying 18% Tax

###### Objective

The objective is to create a new column named final_price, where the final selling price is calculated by adding 18% GST (Tax) to the base price.

Formula:

Final Price = Base Price × 1.18

---

###### Spark Concepts Used

###### withColumn()

The withColumn() function is used to create a new column or replace an existing column in a Spark DataFrame.

###### Column Expressions

Spark allows arithmetic operations directly on DataFrame columns, making transformations efficient and scalable.

---

###### Business Scenario

Suppose an e-commerce company stores the product's base price before tax.

Before displaying the price to customers or generating invoices, GST must be added to calculate the final selling price.

Spark performs this transformation efficiently across millions of records in parallel.

---

###### Production Insight

Instead of modifying the original column, it is considered a best practice to create a new derived column (final_price). This preserves the original data while making the transformed value available for downstream analytics and reporting.

In [0]:
from pyspark.sql.functions import col, round

df_price = (
    df.withColumn(
        "final_price",
        round(col("base_price") * 1.18, 2)
    )
)

print("Final Price Calculated Successfully")

display(
    df_price.select(
        "product_id",
        "base_price",
        "final_price"
    )
)

Final Price Calculated Successfully


product_id,base_price,final_price
P0170,557.35,657.67
P0123,181.35,213.99
P0101,819.62,967.15
P0188,2324.15,2742.5
P0192,276.83,326.66
P0197,855.14,1009.07
P0162,4948.14,5838.81
P0114,1375.32,1622.88
P0217,3737.7,4410.49
P0156,4312.46,5088.7


In [0]:
print("Validation")
print("Total Records :", df_price.count())
print("\nSchema")

df_price.printSchema()

Validation
Total Records : 1000

Schema
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- final_price: double (nullable = true)




##### Q11: What is the Difference Between Transformations and Actions? Provide Two Examples of Each.

###### What are Transformations?

Transformations are operations that create a new DataFrame or RDD from an existing one without immediately executing the computation.

Spark follows Lazy Evaluation, meaning transformations are only recorded in the Lineage Graph (DAG). They are executed only when an Action is called.

Transformations are immutable, meaning they never modify the original DataFrame.

---

###### What are Actions?

Actions are operations that trigger the actual execution of all pending transformations.

When an action is called, Spark optimizes the execution plan using the Catalyst Optimizer and executes the DAG to produce the final result.

Actions usually return:

- A value to the Driver
- Display output
- Write data to storage

---

###### Execution Flow

Read Dataset
      │
      ▼
Transformation
(filter)
      │
      ▼
Transformation
(select)
      │
      ▼
Transformation
(withColumn)
      │
      ▼
DAG Created
      │
      ▼
Action
(show / count / collect / write)
      │
      ▼
Spark Executes the Entire Plan

---

###### Examples

###### Transformations

- filter()
- select()

Other examples:
- withColumn()
- groupBy()
- join()
- drop()
- withColumnRenamed()

---

###### Actions

- show()
- count()

Other examples:

- collect()
- first()
- take()
- write()
- save()

---

###### Key Differences

| Transformation | Action |
|---------------|--------|
| Creates a new DataFrame | Triggers execution |
| Lazy | Eager |
| Builds DAG | Executes DAG |
| Returns DataFrame/RDD | Returns result or writes output |
| No computation immediately | Performs actual computation |

---

###### Production Insight

Spark delays execution of transformations to optimize the complete workflow before processing data.

Instead of executing every operation separately, Spark combines all transformations into a single optimized execution plan, reducing disk I/O, shuffle operations, and execution time.

---

###### Key Takeaway

> Transformations define what should happen to the data, whereas Actions tell Spark when to execute those transformations.

In [0]:
from pyspark.sql.functions import col

df_transformed = (
    df.filter(col("price") > 500)
      .select("product_id", "category", "price")
)

print("Transformations created.")
print("No computation has happened yet because no Action was called.")

Transformations created.
No computation has happened yet because no Action was called.


In [0]:
# trigger 
print("Triggering Action using show()...\n")
df_transformed.show(5, truncate=False)

Triggering Action using show()...

+----------+--------+-------+
|product_id|category|price  |
+----------+--------+-------+
|P0170     |Clothing|584.18 |
|P0101     |Clothing|894.57 |
|P0188     |Toys    |2149.76|
|P0197     |Grocery |830.39 |
|P0162     |Clothing|5086.69|
+----------+--------+-------+
only showing top 5 rows


In [0]:
# trigger another action 
print("Counting Records...")
record_count = df_transformed.count()
print("Total Records:", record_count)

Counting Records...
Total Records: 896


In [0]:
print("Execution Plan")
df_transformed.explain(True)

Execution Plan
== Parsed Logical Plan ==
'Project ['product_id, 'category, 'price]
+- 'Filter '`>`('price, 500)
   +- Relation [order_id#11997,user_id#11998,product_id#11999,category#12000,region#12001,priority#12002,status#12003,base_price#12004,price#12005,amount#12006,old_name#12007] csv

== Analyzed Logical Plan ==
product_id: string, category: string, price: double
Project [product_id#11999, category#12000, price#12005]
+- Filter (price#12005 > cast(500 as double))
   +- Relation [order_id#11997,user_id#11998,product_id#11999,category#12000,region#12001,priority#12002,status#12003,base_price#12004,price#12005,amount#12006,old_name#12007] csv

== Optimized Logical Plan ==
Project [product_id#11999, category#12000, price#12005]
+- Filter (isnotnull(price#12005) AND (price#12005 > 500.0))
   +- Relation [order_id#11997,user_id#11998,product_id#11999,category#12000,region#12001,priority#12002,status#12003,base_price#12004,price#12005,amount#12006,old_name#12007] csv

== Physical Plan 

###### Observation

- The filter() and select() operations are Transformations. They only build the execution plan and do not process the data immediately.
- The show() and count() operations are Actions. They trigger Spark to execute all pending transformations.
- The execution plan displayed using explain(True) demonstrates how Spark combines transformations into an optimized DAG before execution.

---


##### Q12: Load a Parquet File, Remove Rows with Null `user_id`, and Save the Result as a CSV

###### Objective

The objective is to:
1. Read a dataset stored in Parquet format.
2. Remove records where the user_id is null.
3. Save the cleaned dataset as a CSV file.

This represents a common ETL (Extract, Transform, Load) workflow used in production data engineering pipelines.

---

###### Spark Concepts Used

###### spark.read.parquet()

Loads data stored in Parquet format into a Spark DataFrame.

###### filter()

Removes unwanted records based on a condition.

###### write.csv()

Writes the transformed DataFrame to a CSV file.

---

###### Business Scenario

Suppose an e-commerce company stores customer transaction data in Parquet format inside a Data Lake.

Before generating reports, records with missing user_id values must be removed because they cannot be linked to any customer.

The cleaned dataset is then exported as a CSV file for reporting or sharing with another system.

---

###### Production Insight

In production ETL pipelines, data is often stored in **Parquet** because of its efficient columnar storage and faster query performance. After cleaning and validation, the processed data can be exported in CSV format when required by downstream applications or reporting tools.

In [0]:
from pyspark.sql.functions import col

df_parquet = spark.read.parquet(
    "path/to/input"
)

df_clean = df_parquet.filter(
    col("user_id").isNotNull()
)

print("Cleaned DataFrame")

display(df_clean)

Cleaned DataFrame


order_id,user_id,product_id,category,region,priority,status,base_price,price,amount,old_name
ORD00001,U1163,P0170,Clothing,South,High,Cancelled,557.35,584.18,2786.75,Aditya Joshi
ORD00002,U1108,P0123,Clothing,South,Low,Cancelled,181.35,170.43,906.75,Myra Reddy
ORD00003,U1114,P0101,Clothing,West,Medium,Pending,819.62,894.57,2458.86,Sai Verma
ORD00004,U1097,P0188,Toys,East,High,Cancelled,2324.15,2149.76,9296.6,Aditya Mehta
ORD00005,U1075,P0192,Toys,South,Low,Completed,276.83,261.76,830.49,Aditya Rao
ORD00006,U1059,P0197,Grocery,West,Low,Pending,855.14,830.39,2565.42,Aditya Joshi
ORD00007,U1162,P0162,Clothing,West,Medium,Pending,4948.14,5086.69,24740.7,Arjun Kapoor
ORD00008,U1083,P0114,Clothing,North,Medium,Pending,1375.32,1295.82,6876.6,Saanvi Reddy
ORD00009,U1167,P0217,Clothing,East,High,Completed,3737.7,3766.84,18688.5,Myra Bose
ORD00010,U1149,P0156,Clothing,West,High,Completed,4312.46,4013.04,8624.92,Myra Joshi


In [0]:
print("Validation")
print("Total Records Before Cleaning :", df_parquet.count())

print("Total Records After Cleaning :", df_clean.count())

print("Rows Removed :", df_parquet.count() - df_clean.count())

Validation
Total Records Before Cleaning : 1000
Total Records After Cleaning : 979
Rows Removed : 21


In [0]:
# save as csv
df_clean.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", "true") \
.csv("path/to/output")

print("Cleaned CSV saved successfully.")

Cleaned CSV saved successfully.



##### Q13: What is the Difference Between Client Mode and Cluster Mode in Spark Architecture?

###### Introduction

When submitting a Spark application, the Driver Program can run in two different modes:

1. Client Mode
2. Cluster Mode

The primary difference between these modes is where the Driver Program executes.

Both modes use the same Spark APIs and Executors, but they differ in execution location, fault tolerance, and deployment strategy.

---

##### Client Mode

In Client Mode, the Driver Program runs on the machine from which the Spark application is submitted.

        User Machine
     +----------------+
     |     Driver     |
     +----------------+
             │
             ▼
     +----------------+
     | Cluster Manager|
     +----------------+
       │     │      │
       ▼     ▼      ▼
  Executor Executor Executor

###### Characteristics

- Driver runs on the client machine.
- Executors run on worker nodes.
- Easy to debug and monitor.
- If the client machine disconnects, the Spark application also stops.
- Suitable for development, testing, and interactive analysis.

---

###### Cluster Mode

In Cluster Mode, the Driver Program runs inside the cluster along with the Executors.

        User
          │
          ▼
 +----------------------+
 |   Cluster Manager    |
 +----------------------+
          │
          ▼
   +----------------+
   |     Driver     |
   +----------------+
      │      │      │
      ▼      ▼      ▼
 Executor Executor Executor

###### Characteristics

- Driver runs inside the cluster.
- Executors run on worker nodes.
- More reliable than Client Mode.
- The application continues even if the client disconnects.
- Preferred for production workloads.

---

###### Comparison

| Feature | Client Mode | Cluster Mode |
|----------|-------------|--------------|
| Driver Location | Client Machine | Cluster |
| Executors | Worker Nodes | Worker Nodes |
| Fault Tolerance | Lower | Higher |
| Client Dependency | Yes | No |
| Best Use Case | Development & Testing | Production Workloads |
| Reliability | Moderate | High |

---

###### Real-World Example

Suppose a Data Engineer is developing a Spark ETL pipeline in Databricks.

###### During Development

The engineer executes the notebook interactively to test logic and validate outputs.

This is similar to Client Mode, where the Driver is closely connected to the user.

###### In Production

The same ETL pipeline is scheduled to run automatically every night.

Here, Cluster Mode is preferred because the Driver runs inside the cluster, allowing the job to continue even if the user logs out or disconnects.

---

###### Production Insight

Production Spark applications generally use Cluster Mode because it provides:

- Better fault tolerance
- Improved reliability
- No dependency on the client machine
- Better resource management

Client Mode is mainly used for interactive development and debugging.

---

###### Key Takeaway

> Client Mode keeps the Driver on the client machine and is best suited for development and testing.

> Cluster Mode runs the Driver inside the cluster, making it the preferred choice for reliable and scalable production applications.

print("Spark Application Information")

print("Application Name :", spark.sparkContext.appName)

print("Master           :", spark.sparkContext.master)

###### Observation

- The Spark application is running successfully in the Databricks environment.
- Databricks manages the Spark cluster automatically, allowing users to focus on application development rather than cluster configuration.
- Client Mode is generally used for interactive analysis, while Cluster Mode is preferred for production jobs because of its higher reliability and fault tolerance.

###### Databricks Perspective

In Databricks, users typically interact with Spark through notebooks. The platform automatically provisions and manages the cluster, including the Driver and Executors.

During interactive notebook execution, the experience resembles Client Mode because the user works directly with the running Driver. For scheduled Jobs, Databricks launches the Driver within the cluster, providing behavior similar to Cluster Mode for reliable, unattended execution.

This abstraction allows engineers to focus on building ETL pipelines instead of manually managing Spark infrastructure.


##### Q14: Filter a Dataset for Rows Where the Region is 'North' OR the Priority is 'High'

###### Objective

The objective is to filter the dataset and retrieve records where either of the following conditions is satisfied:

- The Region is North
- The Priority is High

This demonstrates the use of logical conditions in Apache Spark.

---

###### Spark Concepts Used

###### filter()

The filter() function is used to retrieve rows based on specified conditions.

###### Logical OR

The OR operator returns records that satisfy at least one of the given conditions.

Unlike the AND operator, both conditions do not need to be true simultaneously.

---

###### Business Scenario

Suppose an e-commerce company wants to generate a report containing:

- All orders from the North region, irrespective of their priority.
- All High Priority orders, regardless of the region.

Such reports help operations teams prioritize urgent deliveries while also monitoring regional performance.

---

###### Production Insight

Applying filters at the beginning of a Spark pipeline minimizes the amount of data processed in later stages. This reduces execution time and improves resource utilization, especially when working with large datasets.

In [0]:
from pyspark.sql.functions import col

filtered_df = (
    df.filter(
        (col("region") == "North") |
        (col("priority") == "High")
    )
)
print("Filtered Dataset")
display(filtered_df)

Filtered Dataset


order_id,user_id,product_id,category,region,priority,status,base_price,price,amount,old_name
ORD00001,U1163,P0170,Clothing,South,High,Cancelled,557.35,584.18,2786.75,Aditya Joshi
ORD00004,U1097,P0188,Toys,East,High,Cancelled,2324.15,2149.76,9296.6,Aditya Mehta
ORD00008,U1083,P0114,Clothing,North,Medium,Pending,1375.32,1295.82,6876.6,Saanvi Reddy
ORD00009,U1167,P0217,Clothing,East,High,Completed,3737.7,3766.84,18688.5,Myra Bose
ORD00010,U1149,P0156,Clothing,West,High,Completed,4312.46,4013.04,8624.92,Myra Joshi
ORD00014,U1135,P0182,Furniture,North,High,Pending,4399.67,4691.51,13199.01,Arjun Sharma
ORD00015,U1061,P0120,Electronics,West,High,Cancelled,3840.88,3555.42,15363.52,Neha Gupta
ORD00017,U1132,P0163,Clothing,North,Medium,Completed,2962.13,2802.24,5924.26,Aarav Verma
ORD00018,U1181,P0158,Electronics,North,Medium,Completed,2595.07,2480.1,10380.28,Kabir Mehta
ORD00019,U1033,P0246,Toys,West,High,Pending,4047.11,3796.52,4047.11,Myra Nair


In [0]:
print("Validation")
print("Total Matching Records :", filtered_df.count())
print("\nSchema")

filtered_df.printSchema()

Validation
Total Matching Records : 496

Schema
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- old_name: string (nullable = true)



###### Observation

- The dataset was successfully filtered using the logical OR (`|`) operator.
- Records belonging to the North region or having High priority were retrieved.
- Spark returned rows satisfying either of the specified conditions.
- Applying filters early in the data processing pipeline reduces unnecessary computation and improves overall performance.



##### Q15: When Exploring a Dataset, Why is it Safer to Use .show(5) Instead of .collect() on a Multi-Terabyte Dataset?

###### Introduction

Apache Spark is a distributed data processing framework where data is processed across multiple **Executor** nodes. When exploring very large datasets, choosing the correct Action is extremely important.

Although both .show() and .collect() trigger Spark execution, they behave very differently in terms of memory usage and performance.

---

###### What is .show()?

The .show() function displays only a limited number of rows (20 by default or the specified number).

Example:
df.show(5)

Spark retrieves only the required rows and prints them on the Driver.

This makes .show() suitable for:
- Data Exploration
- Debugging
- Schema Validation
- Quick Inspection

---

###### What is .collect()?

The .collect() function retrieves every row from all Executor nodes and transfers the complete dataset to the Driver Program as a Python list.

Example:
rows = df.collect()

This operation should only be used for small datasets.

---

###### Internal Working

###### Using .show(5)
Executors

↓

Read Partitions

↓

Return Only 5 Rows

↓

Driver

↓

Display Output

Only a small amount of data is transferred to the Driver.

---

###### Using .collect()

Executors

↓

Read Entire Dataset

↓

Transfer ALL Rows

↓

Driver Memory

↓

Python List

Every record is copied to the Driver, regardless of dataset size.

---

###### Why is .collect() Dangerous?

Suppose a Spark cluster is processing a 5 TB dataset.

If we execute:
df.collect()


Spark attempts to move the entire dataset from all Executors to the Driver.

Possible consequences:
- Driver OutOfMemoryError
- Application Failure
- High Network Traffic
- Slow Execution
- Cluster Instability

---

###### Comparison

| Feature | .show(5) | .collect() |
|----------|------------|--------------|
| Rows Returned | Limited | Entire Dataset |
| Memory Usage | Low | Very High |
| Network Traffic | Minimal | Very High |
| Suitable for Large Data | Yes | No |
| Risk of Driver Failure | Very Low | High |

---

###### Real-World Example

Suppose an online retail company stores **10 TB** of transaction data.

A Data Engineer wants to verify whether the dataset has loaded correctly.

###### Recommended
df.show(5)

Only five records are displayed.

###### Not Recommended

df.collect()

Spark attempts to transfer all 10 TB of data to the Driver, which may crash the application due to insufficient memory.

---

###### Production Insight

In production ETL pipelines, Data Engineers almost always use:

- .show()
- .limit()
- .take()
- .sample()

for inspecting large datasets.

The .collect() function should only be used when the resulting dataset is known to be very small.

---
> .show(5) is safe because it retrieves only a few records for inspection.

> .collect() transfers the entire dataset to the Driver, making it unsuitable for very large datasets due to high memory consumption and the risk of application failure.

In [0]:
print("Using show(5)")
df.show(5, truncate=False)

Using show(5)
+--------+-------+----------+--------+------+--------+---------+----------+-------+-------+------------+
|order_id|user_id|product_id|category|region|priority|status   |base_price|price  |amount |old_name    |
+--------+-------+----------+--------+------+--------+---------+----------+-------+-------+------------+
|ORD00001|U1163  |P0170     |Clothing|South |High    |Cancelled|557.35    |584.18 |2786.75|Aditya Joshi|
|ORD00002|U1108  |P0123     |Clothing|South |Low     |Cancelled|181.35    |170.43 |906.75 |Myra Reddy  |
|ORD00003|U1114  |P0101     |Clothing|West  |Medium  |Pending  |819.62    |894.57 |2458.86|Sai Verma   |
|ORD00004|U1097  |P0188     |Toys    |East  |High    |Cancelled|2324.15   |2149.76|9296.6 |Aditya Mehta|
|ORD00005|U1075  |P0192     |Toys    |South |Low     |Completed|276.83    |261.76 |830.49 |Aditya Rao  |
+--------+-------+----------+--------+------+--------+---------+----------+-------+-------+------------+
only showing top 5 rows


In [0]:
print("Using collect()")
rows = df.limit(5).collect()

for row in rows:
    print(row)

Using collect()
Row(order_id='ORD00001', user_id='U1163', product_id='P0170', category='Clothing', region='South', priority='High', status='Cancelled', base_price=557.35, price=584.18, amount=2786.75, old_name='Aditya Joshi')
Row(order_id='ORD00002', user_id='U1108', product_id='P0123', category='Clothing', region='South', priority='Low', status='Cancelled', base_price=181.35, price=170.43, amount=906.75, old_name='Myra Reddy')
Row(order_id='ORD00003', user_id='U1114', product_id='P0101', category='Clothing', region='West', priority='Medium', status='Pending', base_price=819.62, price=894.57, amount=2458.86, old_name='Sai Verma')
Row(order_id='ORD00004', user_id='U1097', product_id='P0188', category='Toys', region='East', priority='High', status='Cancelled', base_price=2324.15, price=2149.76, amount=9296.6, old_name='Aditya Mehta')
Row(order_id='ORD00005', user_id='U1075', product_id='P0192', category='Toys', region='South', priority='Low', status='Completed', base_price=276.83, price=

###### Observation

- The show(5) function displayed only the first five records without transferring the complete dataset to the Driver.
- The collect() function returns the data as a Python list. To avoid excessive memory usage, it was demonstrated on a limited subset of the dataset using limit(5).
- On large datasets, show() is preferred for exploration because it is memory-efficient and reduces network traffic.